# Introspecting a Datasource and a Model via MCP

SLayer's MCP (Model Context Protocol) server exposes tools like `query`, `models_summary`, `create_model`, and — the focus of this notebook — **`describe_datasource`** and **`inspect_model`**, which together tell an LLM agent everything it needs to know before issuing a query.

`inspect_model` returns a complete markdown view of a semantic model: metadata, all filters, columns, measures, custom aggregations, direct joins, and a data-profile table run against the live database. The Columns table gets a `sampled` column that carries the per-column data profile inline — distinct values for string/boolean columns, `min .. max` for number/date/time columns. Every table auto-prunes columns that are empty for every row and collapses to a comma-separated backticked list when only one column survives.

It takes just two parameters:

| Parameter | Default | Effect |
|-----------|---------|--------|
| `model_name` | — required — | Name of the model to inspect |
| `num_rows` | `3` | Max rows in the data-profile table |

### A note on transport

`slayer mcp` runs over **stdio**, not HTTP, so a literal `curl` can't reach it. In this notebook we invoke each tool through the **in-process Python MCP client** — `await mcp.call_tool(name=..., arguments=...)` — which issues the same JSON-RPC `tools/call` request a desktop MCP client would issue over stdio. The REST API (`slayer serve` / port 5143) is a separate surface and is not used here.

We pick **`items`** as the subject of `inspect_model` because it has a rich join graph in Jaffle Shop: direct joins to `orders` and `products`, plus multi-hop reach to `customers` and `stores` through `orders`.

See also: [Joins](../05_joins/joins.md) · [Auto Ingestion](../03_auto_ingest) · [Multi-Stage Queries](../06_multistage_queries)

In [1]:
import json
import os
import sys
import warnings
from pathlib import Path

# FastMCP emits a benign PydanticJsonSchemaWarning when it builds the tool
# JSON schemas; silence it so the tool listing below stays clean.
warnings.filterwarnings("ignore", message=".*not JSON serializable.*")

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()
print(f"Loaded {len(models)} models: {sorted(m.name for m in models)}")

Loaded 8 models: ['customers', 'items', 'orders', 'products', 'regions', 'stores', 'supplies', 'tweets']


## Why `items`?

During ingestion SLayer creates one `ModelJoin` per FK on the source table. `items` has direct FKs to `orders` and `products`. Multi-hop targets like `customers` and `stores` are reached at query time by walking through `orders`'s own joins:

```
items ──► orders ──► customers
      │         └──► stores
      └──► products
```

The next cell prints the raw `ModelJoin` entries on `items` and `orders`.

In [2]:
items = next(m for m in models if m.name == "items")
orders_m = next(m for m in models if m.name == "orders")

print("items joins (direct FKs only):")
for j in items.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<10} ON {src:<22} = {tgt}")

print("\norders joins (direct FKs only):")
for j in orders_m.joins:
    src, tgt = j.join_pairs[0]
    print(f"  -> {j.target_model:<10} ON {src:<22} = {tgt}")

print("\nMulti-hop: items reaches customers/stores via orders.joins")

items joins (direct FKs only):
  -> orders     ON order_id               = id
  -> products   ON sku                    = sku

orders joins (direct FKs only):
  -> customers  ON customer_id            = id
  -> stores     ON store_id               = id

Multi-hop: items reaches customers/stores via orders.joins


## Wiring up the MCP client

We instantiate the **MCP** server (from `slayer.mcp.server`, _not_ the REST app in `slayer.api.server`) against the same storage the engine just loaded. `create_mcp_server` returns a `FastMCP` instance; its `list_tools()` and `call_tool(name, arguments)` methods are exactly what stdio clients invoke under the hood.

Over stdio, each `call_tool` request serialises to a JSON-RPC frame like this (the `params` payload is what would be the POST body if SLayer's MCP ran on HTTP):

```json
{
  "jsonrpc": "2.0", "id": 1, "method": "tools/call",
  "params": {
    "name": "inspect_model",
    "arguments": {"model_name": "items", "num_rows": 10}
  }
}
```

In [3]:
from slayer.mcp.server import create_mcp_server

mcp = create_mcp_server(storage=storage)

tools = await mcp.list_tools()
print(f"MCP server exposes {len(tools)} tools:")
for t in tools:
    marker = "<-- targets" if t.name in ("describe_datasource", "inspect_model") else ""
    print(f"  - {t.name} {marker}")

inspect_tool = next(t for t in tools if t.name == "inspect_model")
print("\ninspect_model input schema:")
print(json.dumps(inspect_tool.inputSchema, indent=2))

MCP server exposes 20 tools:
  - query 
  - models_summary 
  - inspect_model <-- targets
  - inspect 
  - create_model 
  - edit_model 
  - create_datasource 
  - list_datasources 
  - describe_datasource <-- targets
  - edit_datasource 
  - delete_model 
  - validate_models 
  - recommend_root_model 
  - delete_datasource 
  - ingest_datasource_models 
  - set_datasource_priority 
  - get_datasource_priority 
  - save_memory 
  - forget_memory 
  - search 

inspect_model input schema:
{
  "properties": {
    "model_name": {
      "description": "Model name.",
      "title": "Model Name",
      "type": "string"
    },
    "num_rows": {
      "default": 3,
      "description": "Sample rows.",
      "title": "Num Rows",
      "type": "integer"
    },
    "show_sql": {
      "default": false,
      "description": "Include generated SQL.",
      "title": "Show Sql",
      "type": "boolean"
    },
    "format": {
      "default": "markdown",
      "description": "\"markdown\" or \"json\"."

## Step 1 — Introspect the datasource

Before drilling into a specific model, a well-behaved agent verifies the datasource: is it reachable, what's its type, what schemas are available, and what tables live there? The MCP `describe_datasource` tool covers all of that in one call. Pass `list_tables=False` to skip the table-listing block if you just want the connection probe; pass `schema_name="..."` to scope the table listing.

The Jaffle Shop fixture registered a DuckDB datasource named `jaffle_shop` when we called `ensure_jaffle_shop()`, so let's inspect it.

In [4]:
describe_args = {"name": "jaffle_shop"}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(describe_args, indent=2))

content, _ = await mcp.call_tool(name="describe_datasource", arguments=describe_args)
# Show any absolute DB path relative to the repo root so the committed output
# carries no local username / workstation path.
repo_root = Path.cwd().parents[2]
print("\n=== describe_datasource response ===\n")
print(content[0].text.replace(f"{repo_root}/", ""))

Arguments (the 'curl POST body' equivalent):
{
  "name": "jaffle_shop"
}

=== describe_datasource response ===

Datasource: jaffle_shop
Type: duckdb
Database: docs/examples/jaffle_data/demo/jaffle_shop.duckdb

Connection: OK
Available schemas: jaffle_shop.main, system.information_schema, system.main, temp.main

Tables (8):
  - customers
  - items
  - orders
  - products
  - regions
  - stores
  - supplies
  - tweets

Use ingest_datasource_models to create models from these tables.


## Step 2 — Survey the datasource's models with `models_summary`

Now that we know the datasource works, `models_summary` gives a compact markdown overview of every model registered against it: name, description, its **column count**, its **measure names**, and the models it joins to. It's deliberately lighter than `inspect_model` — no types, no distinct values, no sample data — so an agent can scan N models without making N heavy calls.

In [5]:
summary_args = {"datasource_name": "jaffle_shop"}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(summary_args, indent=2))

content, _ = await mcp.call_tool(name="models_summary", arguments=summary_args)
print("\n=== models_summary response (markdown) ===\n")
print(content[0].text)

Arguments (the 'curl POST body' equivalent):
{
  "datasource_name": "jaffle_shop"
}

=== models_summary response (markdown) ===

# Datasource: `jaffle_shop` — 9 model(s)

## `customers`
Customers of the Jaffle Shop — one row per person.
Columns: 2
Measures: customer_count
Joins to: `orders`, `tweets`

## `items`
Order line items — one row per unit sold on an order.
Columns: 3
Measures: units_sold
Joins to: `orders`, `products`

## `orders`
Customer orders — one row per order. Monetary amounts are in dollars. The fact table at the center of the demo: joins to customers and stores, and is referenced by items.
Columns: 9
Measures: total_revenue, net_sales, tax_collected, order_count, unique_customers, avg_order_value, effective_tax_rate, sales_weighted_aov
Joins to: `customers`, `item_skus`, `items`, `stores`

## `products`
Menu items sold at the Jaffle Shop — jaffles and beverages.
Columns: 6
Measures: product_count, avg_price
Joins to: `ingredients`, `items`, `supplies`

## `regions`
Sa

## Step 3 — Drill into one model with `inspect_model`

We've picked `items` from the `models_summary` output above. `inspect_model` returns a single markdown document — we print it verbatim rather than parsing as JSON.

In [6]:
arguments = {"model_name": "items", "num_rows": 10}
print("Arguments (the 'curl POST body' equivalent):")
print(json.dumps(arguments, indent=2))

content, _ = await mcp.call_tool(name="inspect_model", arguments=arguments)
markdown = content[0].text

print(f"\nResponse length: {len(markdown):,} characters")
print("\n=== inspect_model response (markdown) ===\n")
print(markdown)

Arguments (the 'curl POST body' equivalent):
{
  "model_name": "items",
  "num_rows": 10
}

Response length: 2,277 characters

=== inspect_model response (markdown) ===

# Model: `items`

Order line items — one row per unit sold on an order.

- **data_source:** `jaffle_shop`
- **sql_table:** `items`
- **row_count:** 984,271

## Columns (3)

| name | type | primary_key | allowed_aggregations | label | description | sampled |
| --- | --- | --- | --- | --- | --- | --- |
| id | TEXT | yes | all | Item ID | — | — |
| order_id | TEXT | — | all | Order | Order this line belongs to. | 001424df-f7b4-451d-b7b3-960b3d40fecb, 00180913-3b87-40a7-a606-b2783d787653, 001853c9-e840-481d-b028-90e43867b29a, 001b65eb-569c-4d37-ab14-dd478a19fcb8, 00200ba3-9e71-4b13-b4b1-b0bff2b73927, 0035668e-9634-4ba6-a958-9ad056bd289d, 00452aa6-31d4-4e02-924d-2b7141910d0c, 0054819b-4d54-4312-9ea7-f16d7b6c2853, 0081883a-1d39-4d30-8769-7565015c6b67, 009ed227-a12a-480f-bc53-bf5d6c157915, 00a207fb-6072-484e-bbdf-1d12756586f8

## What we just got

The exploration flow was `describe_datasource` → `models_summary` → `inspect_model` — exactly the sequence an MCP-driven agent would run before composing its first `query` call.

`describe_datasource` gave us the datasource type (`duckdb`), the file backing it, a successful connection probe, the available schemas, and the list of tables in one call. `models_summary` returned a brief overview of every model in that datasource. `inspect_model` returned a single markdown document covering:

- `# Model: <name>` header and description
- Metadata bullets: `data_source`, `sql_table`, `default_time_dimension`, `row_count`
- `## SQL` fenced block when the model has a custom SQL expression (query-backed models)
- `## Filters (model-level)` — always-applied WHERE conditions
- `## Columns` — markdown table (`name`, `type`, `primary_key`, `allowed_aggregations`, `label`, `description`) with a `sampled` column that embeds the per-column data profile: distinct values (e.g. the SKUs for `sku`) for string/boolean columns, and `<min> .. <max>` for number/date/time columns
- `## Measures` — markdown table with `name`, `formula`, `label`, `description`
- `## Aggregations` — custom `Aggregation` definitions with their formulas
- `## Joins` — target model, join pairs, and cardinality
- `## Data Profile` — a real `SlayerQuery` executed through `engine.execute`, with `count(*)` plus one aggregation per measure (`avg` when allowed, otherwise the first permitted aggregation)

Every markdown table in the response auto-prunes empty columns. When exactly one column survives pruning, the table collapses to a comma-separated backticked list — which is why a column or measure table from `inspect_model` on an auto-ingested model (where descriptions are often absent) often renders as a dense list of names rather than a two-column table of mostly em-dashes.

Other MCP tools that complement these for an exploring agent: `list_datasources` (enumerate datasources) and `models_summary(datasource_name="...")` — the latter returns a compact markdown summary of every model in one datasource (name + description + column count + measure names + list of joined model names), much lighter than `inspect_model` and great for picking the right model to drill into next.